In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import re
import ast

In [2]:
# ==========================================================
# USER SETTINGS (EDIT THESE)
# ==========================================================
CHIRPS_DIR = Path(r"C:\Users\SID-DRW\Downloads\precip_tifs")       # contains CHIRPS_Brahmaputra_YYYY.tif
MASK_DIR  = Path(r"C:\Users\SID-DRW\Downloads\upstream_masks")    # contains upstream_mask_<r>.tif
OUT_DIR   = Path(r"C:\Users\SID-DRW\Downloads\monthly_CI_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "1984-01-01"
END_DATE   = "2024-12-31"

WET_THRESHOLD = 0.1  # mm/day
# ==========================================================


# --------------------------
# Helper: keep only valid yearly CHIRPS stacks
# --------------------------
def is_yearly_chirps_stack(tif_path: Path) -> bool:
    # Must be exactly CHIRPS_Brahmaputra_YYYY.tif
    m = re.match(r"CHIRPS_Brahmaputra_(\d{4})\.tif$", tif_path.name)
    if not m:
        return False

    try:
        with rasterio.open(tif_path) as ds:
            # yearly stacks should have many bands (~365)
            if ds.count < 50:
                return False

            desc = ds.descriptions
            if (desc is None) or any(d is None for d in desc):
                return False

            # Reject common non-CHIRPS rasters like HYBAS_ID
            if str(desc[0]).upper() == "HYBAS_ID":
                return False

            # Must parse as YYYYMMDD (at least first+last)
            pd.to_datetime([desc[0], desc[-1]], format="%Y%m%d")
            return True
    except Exception:
        return False


# --------------------------
# Monthly CI (Lorenz/Gini) over daily values
# --------------------------
def concentration_index_monthly(pr: pd.Series, wet_threshold: float = 0.1) -> pd.Series:
    """
    Monthly precipitation concentration index (CI) (0–1),
    computed for each month from daily precip values.
    Index is month-start timestamps.
    """
    pr = pr.dropna()
    if pr.empty:
        return pd.Series(dtype=float, name="CI")

    results = {}

    for month_start, s in pr.groupby(pd.Grouper(freq="MS")):
        if month_start is pd.NaT:
            continue

        wet = s[s > wet_threshold].values
        if wet.size < 2 or wet.sum() == 0:
            results[month_start] = np.nan
            continue

        x = np.sort(wet)
        n = len(x)
        cumx = np.cumsum(x)

        p = np.arange(1, n + 1) / n
        L = cumx / cumx[-1]

        auc = np.trapz(np.r_[0, L], np.r_[0, p])
        gini = 1 - 2 * auc

        results[month_start] = float(gini)

    return pd.Series(results, name="CI").sort_index()


# --------------------------
# Build daily upstream-area mean precip series for one centroid mask
# --------------------------
def daily_mean_series_for_mask(chirps_files, mask_path, start_date, end_date) -> pd.Series:
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    with rasterio.open(mask_path) as mds:
        mask = mds.read(1).astype(np.uint8)
        mask_bool = mask == 1
        m_crs = mds.crs
        m_transform = mds.transform
        m_shape = (mds.height, mds.width)

    if mask_bool.sum() == 0:
        return pd.Series(dtype=float, name="precip_mean")

    all_dates = []
    all_vals = []

    for tif in chirps_files:
        with rasterio.open(tif) as ds:
            # Hard alignment checks
            if (ds.height, ds.width) != m_shape:
                raise ValueError(f"Shape mismatch: {tif.name} {ds.height, ds.width} vs mask {m_shape}")
            if ds.crs != m_crs:
                raise ValueError(f"CRS mismatch: {tif.name} {ds.crs} vs mask {m_crs}")
            if ds.transform != m_transform:
                raise ValueError(f"Transform mismatch: {tif.name} {ds.transform} vs mask {m_transform}")

            desc = ds.descriptions
            if desc is None or any(d is None for d in desc):
                raise ValueError(f"{tif.name}: Missing band descriptions (expected YYYYMMDD per band).")

            # Parse band dates (YYYYMMDD)
            dates = pd.to_datetime(list(desc), format="%Y%m%d")

            # Skip whole file if out of requested date range
            if dates.max() < start or dates.min() > end:
                continue

            data = ds.read(masked=True).filled(np.nan)  # (bands, y, x)

        # Time window
        keep = (dates >= start) & (dates <= end)
        if not np.any(keep):
            continue

        data = data[keep, :, :]
        dates = dates[keep]

        # Apply upstream mask
        data[:, ~mask_bool] = np.nan

        # Mean over space
        daily_mean = np.nanmean(data, axis=(1, 2))

        all_dates.append(dates.values)
        all_vals.append(daily_mean)

    if not all_dates:
        return pd.Series(dtype=float, name="precip_mean")

    idx = pd.DatetimeIndex(np.concatenate(all_dates))
    vals = np.concatenate(all_vals).astype(float)

    return pd.Series(vals, index=idx, name="precip_mean").sort_index()


# ==========================================================
# MAIN
# ==========================================================
def main():
    # 1) Find all candidate CHIRPS files and filter to valid yearly stacks
    chirps_all = sorted(CHIRPS_DIR.glob("CHIRPS_Brahmaputra_*.tif"))
    chirps_files = [p for p in chirps_all if is_yearly_chirps_stack(p)]

    print("All matching CHIRPS_Brahmaputra_*.tif:")
    print([p.name for p in chirps_all])
    print("\nKept as valid yearly stacks:")
    print([p.name for p in chirps_files])

    if not chirps_files:
        raise FileNotFoundError(
            f"No valid yearly CHIRPS stacks found in {CHIRPS_DIR}. "
            "Expected files named CHIRPS_Brahmaputra_YYYY.tif with YYYYMMDD band descriptions."
        )

    # 2) Find centroid masks
    mask_files = sorted(MASK_DIR.glob("upstream_mask_*.tif"))
    if not mask_files:
        raise FileNotFoundError(f"No upstream_mask_*.tif found in {MASK_DIR}")

    print(f"\nFound {len(mask_files)} masks, computing monthly CI per centroid...")

    summary_rows = []

    for mask_path in mask_files:
        # Accept filenames like upstream_mask_2.tif or upstream_mask_r2.tif
        r_label = mask_path.stem.replace("upstream_mask_", "")  # e.g. "r2" or "2"

        print(f"\n=== Centroid {r_label} ===")
        pr = daily_mean_series_for_mask(
            chirps_files=chirps_files,
            mask_path=mask_path,
            start_date=START_DATE,
            end_date=END_DATE,
        )

        out_csv = OUT_DIR / f"monthly_CI_{r_label}.csv"

        if pr.empty:
            print("  No data in requested date range -> writing empty CSV")
            pd.DataFrame(columns=["r", "month", "CI", "CI_normalized"]).to_csv(out_csv, index=False)
            summary_rows.append({"r": r_label, "n_days": 0, "n_months": 0, "out_csv": str(out_csv)})
            continue

        ci_monthly = concentration_index_monthly(pr, wet_threshold=WET_THRESHOLD)

        # Normalize within centroid (same idea as your yearly normalization)
        if ci_monthly.notna().sum() >= 2 and (ci_monthly.max() != ci_monthly.min()):
            ci_norm = (ci_monthly - ci_monthly.min()) / (ci_monthly.max() - ci_monthly.min())
        else:
            ci_norm = pd.Series(np.nan, index=ci_monthly.index, name="CI_normalized")
        ci_norm.name = "CI_normalized"

        out_df = pd.DataFrame({
            "r": r_label,
            "month": ci_monthly.index.strftime("%Y-%m"),
            "CI": ci_monthly.values,
            "CI_normalized": ci_norm.values
        })

        out_df.to_csv(out_csv, index=False)
        print(f"  Wrote: {out_csv}")

        summary_rows.append({
            "r": r_label,
            "n_days": int(pr.notna().sum()),
            "n_months": int(ci_monthly.shape[0]),
            "out_csv": str(out_csv),
        })

    summary = pd.DataFrame(summary_rows)
    summary_path = OUT_DIR / "monthly_CI_run_summary.csv"
    summary.to_csv(summary_path, index=False)
    print(f"\nWrote run summary: {summary_path}")


if __name__ == "__main__":
    main()


All matching CHIRPS_Brahmaputra_*.tif:
['CHIRPS_Brahmaputra_1984.tif', 'CHIRPS_Brahmaputra_1985.tif', 'CHIRPS_Brahmaputra_1986.tif', 'CHIRPS_Brahmaputra_1987.tif', 'CHIRPS_Brahmaputra_1988.tif', 'CHIRPS_Brahmaputra_1989.tif', 'CHIRPS_Brahmaputra_1990.tif', 'CHIRPS_Brahmaputra_1991.tif', 'CHIRPS_Brahmaputra_1992.tif', 'CHIRPS_Brahmaputra_1993.tif', 'CHIRPS_Brahmaputra_1994.tif', 'CHIRPS_Brahmaputra_1995.tif', 'CHIRPS_Brahmaputra_1996.tif', 'CHIRPS_Brahmaputra_1997.tif', 'CHIRPS_Brahmaputra_1998.tif', 'CHIRPS_Brahmaputra_1999.tif', 'CHIRPS_Brahmaputra_2000.tif', 'CHIRPS_Brahmaputra_2001.tif', 'CHIRPS_Brahmaputra_2002.tif', 'CHIRPS_Brahmaputra_2003.tif', 'CHIRPS_Brahmaputra_2004.tif', 'CHIRPS_Brahmaputra_2005.tif', 'CHIRPS_Brahmaputra_2006.tif', 'CHIRPS_Brahmaputra_2007.tif', 'CHIRPS_Brahmaputra_2008.tif', 'CHIRPS_Brahmaputra_2009.tif', 'CHIRPS_Brahmaputra_2010.tif', 'CHIRPS_Brahmaputra_2011.tif', 'CHIRPS_Brahmaputra_2012.tif', 'CHIRPS_Brahmaputra_2013.tif', 'CHIRPS_Brahmaputra_2014.tif',

C:\Users\SID-DRW\AppData\Local\Temp\ipykernel_13176\2139347536.py:77: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(np.r_[0, L], np.r_[0, p])


  Wrote: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_r1.csv

=== Centroid r10 ===


C:\Users\SID-DRW\AppData\Local\Temp\ipykernel_13176\2139347536.py:77: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(np.r_[0, L], np.r_[0, p])


  Wrote: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_r10.csv

=== Centroid r11 ===


...

C:\Users\SID-DRW\AppData\Local\Temp\ipykernel_13176\2139347536.py:77: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(np.r_[0, L], np.r_[0, p])


  Wrote: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_r7.csv

=== Centroid r8 ===


C:\Users\SID-DRW\AppData\Local\Temp\ipykernel_13176\2139347536.py:77: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(np.r_[0, L], np.r_[0, p])


  Wrote: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_r8.csv

=== Centroid r9 ===
  Wrote: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_r9.csv

Wrote run summary: C:\Users\SID-DRW\Downloads\monthly_CI_outputs\monthly_CI_run_summary.csv


C:\Users\SID-DRW\AppData\Local\Temp\ipykernel_13176\2139347536.py:77: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(np.r_[0, L], np.r_[0, p])
